# Final Project Writeup
#### Maxwell Rodgers

## Overview
This project implements a random and minimax agent to play the abstract strategy game Tak, as well as a human interface to play against these agents.

### About Tak
Tak is an abstract strategy game in which players take turns placing or moving their pieces. The board is between $3 \times 3$ and $8 \times 8$. The goal is to make a 'road' of orthogonally adjacent pieces from 
any side of the board to the opposite side, similar to Hex. The game can also end when a player runs out of
pieces or there are no empty spaces left on the board. In this case, the player who controlls the most
squares wins. There is a handicap for the first player in the form of a 'komi' (taken from the game Go).
The komi is a value that is added to the count of the 2nd players stones (this does not matter for games that end in a road).
The game can be a tie if the komi is a whole number. Full rules can be found at 
https://ustak.org/play-beautiful-game-tak/

## Explanation of code
The `Game` class contains the main loop of the game and a few other functions. In Tak, players are supposed to place one of their opponents pieces to start the game. I did not want to have to implement this for the minimax agent so I included this functionality but it is not enabled in the code. The `game.play()` function calls the `game.turn()` function until one of the end conditions is reached. This function calls each player to get a move, and then uses the `board.move()` method to make that move on the board. 

There are 3 player types, one for the random agent, one for the minimax agent and one for the human interface. Each of them implements a `get_move()` method. For the random agent, this just gets a list of all possible moves (by looping through each square and checking for all possible movements and placements) and chooses one at random. For the human player class, it gets input from the user and converts that into a `Move` object to return. For the minimax agent, this runs the minimax algorithm (with or without alpha-beta pruning) to find a move to return. (I will discuss this in more detail later)

I did not use any libraries or frameworks in this project other that what is provided in the Python standard library.

## Testing

### Game Results

In my intermediate report I described how I tested the random player. This testing showed that there was a slight first player advantage. I theorized that this advantage would get larger with more proficient agents because fewer games would end with one player just placing all their pieces or the board filling up. 

Tak is a somewhat complex game: Even with alpha-beta pruning, each level of the game tree on a $3 \times 3$ board had between 4 and 16 times as many nodes as the previous layer. Without alpha-beta pruning this was 16 to 25 times as many. Due to this I only did a few different test 10 times each for the minimax agents.

For boards of size 3, 4, and 5, and depths of 1 through 3 I did 4 different tests:
- Random vs Minimax
- Minimax vs Minimax
- Minimax vs Minimax w/ depth+1
- Minimax w/ depth+1 vs Minimax

I did all of these tests twice, once with a decay multiplyer of 0.99 and once with a multiplyer of 1.

The raw data for these tests is included in the zip file in the `test_99.txt` and `test_100.txt` files, but I will give my interpretation of the results below.

To start, I did not observe a single game in which a random agent beat a minimax agent, even with minimax of only depth 1.

There continued to be a first player advantage with the 1st player winning more often than the 2nd player when they had the same depth. 
When equally matched there seemed to be about a 70% chance that the first player would win and a negligible chance of a tie (the data is very noisy however so take these numbers with a grain of salt). Based on this, I was correct in guessing that the first player advantage would increase with better agents. Additionally, when the players were unevenly matched, the agent with higher depth won more often as player 1 than as player 2. 

There was not a significant difference between the results for the different decay multipliers

### Performance

I tested the performance of the minimax algorithm with and without alpha-beta pruning by measuring the time for 100 games of Tak on a $3 \times 3$ board with 2 agents both with max seach depth of 3.

For minimax *with* alpha-beta pruning, this took about 74 seconds, averaging 50ms per move.

For minimax *without* alpha-beta pruning, this took about 35 minutes, averaging 1451ms per move.

This means that alpha-beta pruning alone gave about a $29\times$ speed increase over regular minimax.


## Explanation of Minimax agent (Extra Credit)

The minimax agent uses the minimax algorith (with or without alpha-beta pruning) to find an optimal move. The algorithm does this by doing a depth-first search of the game tree. At each node after evaluating each possible move, it takes the highest valued move if it is the player who called it's turn or the lowest valued move if it is their opponents turn. This simulates the opponent making the best possible move for them in every position, and essentially allows the algorithm to return the best move for which the opponent has the worst response. Alpha-Beta pruning gives a way to improve this. We keep track of the score of the best move we have found (alpha) and the best move our oppenent has (beta). We then know that if we have a better response than beta, our opponent would prefer instead to do whatever move got them beta, so we can stop evaluating this path (and vice versa for pruning nodes where it is our opponent's turn).

To implement this, I used the same function that generated all the possible moves for the random agents and used it to get every move. I then looped over this array of moves and recursively called minimax on each one. I then checked and updated alpha and beta and repeated. The leaf nodes need the board so that they can evaluate it and return a score, so I needed to pass the board as well. Initially I was copying the board during each iteration of minimax to prevent it from changing but this was very inefficient. Instead, I wrote a new method for `Board`, `board.unmove()` that undoes a move.  The `unmove()` function allows the real board to be mutated and then reverted without having to copy anything which provided significant speedups.

Once the algorithm reaches the leaf nodes, we need to evaluate the board. I chose a very simple evaluation function based on the rules of the game. Since it is possible for a player to win simply by having more flatstones on top of stacks, I repurposed the scoring function to get the score for each player and then subtracted the score of the opponent from the current player. This incentivises controlling as much area as possible and preventing your opponent from controlling area at the same time. I also needed a way to incentivise building roads. The way that I did this was just by evaluating any board that had a road for the current player as 100 and for the opponent -100. I considered changing this to be somehow based on the size of the board (since the number of pieces scales up based on the size of the board) but it did not seem to make a difference when I tried. There is a lot more that can be done here in terms of searching for boards that are 1 away from making roads, disincentivising making repetative moves or very tall stacks (this is a bigger problem if the moves are checked in the same order each time) and so on.

There are  a few ideas that I didn't get arround to implementing that I would like to mention as well. I tried implementing Zobrist hashing which is a method that I came across while looking at the chess programming wiki, but the hash that I created was too costly and actually made the minimax slower. The reason that the hash was too slow was that it was invariant under rotation of the board. Since the orientation of the board doesn't matter, I was trying to implement a way to reduce he number of mirrored or rotated boards that were checked.

## Conclusion
I learned quite a bit about python from this project. Before this I only had a vauge understanding of python, mostly having used it for math related numpy and matplotlib work. Through being forced to optimize much of my code, I understand much more about how python works under the hood. I also have learned about a number of optimization techniques and aspects of the standard library that I did not have experience with before. For example, to speed up testing I learned about multiprocessing and multithreading (and the difference between them!).

This project has also given me significant insight into the minimax algorith with alpha-beta pruning, and more broadly into implementing agents in adversarial environments. Before this project I did not exactly understand alpha-beta pruning. In fact, I thought that alpha and beta were global variables and attempted to implement them this way before realizing that this was not correct. Implementing these algorithms greatly improved my understanding of them and has made me much more comfortable working with them.

## Code and Examples

Below is the code for the project. It is split up into a few sections. The first section contains the imports and the global constants. These can be changed to modify the behaviour of the program (see comments for details about what each one does). The next sections contain the code for each of the classes for the game. The final section contains several games that can be run to demonstrate the capabilities of the game.

Make sure to run all cells again after any change, I have encountered a number of errors that were fixed by doing this.

### Imports and Consts

In [41]:
from enum import Enum
import functools
from math import inf
import random
from typing import override
import cProfile
import pstats


#number of stones a player gets based on board size
CAPSTONES = {3:0, 4:0, 5:1, 6:1, 7:2, 8:2}
NORMAL_STONES = {3:10, 4:15, 5:21, 6:30, 7:40, 8:50}

#default depth for minimax
DEPTH = 3 
#factor to multiply values by
MINIMAX_DECAY = 0.99
#Whether or not to do alpha-beta pruning
ALPHA_BETA = True 
#Whether or not to shuffle the generated moves before evaluating them
RANDOM_MOVE_ORDERING = True

#ANSI color escapes
B_ON_W = "\033[30;107m"
W_ON_B = "\033[97;40m"
RESET = "\033[0m"

#max width of board (in characters)
TERM_WIDTH = 50


class Color(Enum):
    BLACK = 0
    WHITE = 1

class PieceType(Enum):
    FLATSTONE     = 0
    STANDINGSTONE = 1
    CAPSTONE      = 2

class Direction(Enum):
    UP    = 0
    RIGHT = 1
    DOWN  = 2
    LEFT  = 3

DIRECTIONS = list(Direction)


### Helper functions

In [42]:

# General purpose menu that takes a dictionary as input, list the keys as options and returns the corresponding entry
# while checking for invalid input.
def menu(params, message: str | None = None):                                               # pyright: ignore[reportUnknownParameterType, reportMissingParameterType]
    keys = list(params.keys())                                                          # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType, reportUnknownArgumentType]
    numkeys= len(keys)                                                                        # pyright: ignore[reportUnknownArgumentType]
    while True:
        if message is not None:
            print(message)
        else:
            print("Choose one of the following:")
        
        #print options
        for i,p in enumerate(keys):                                                                # pyright: ignore[reportUnknownVariableType, reportUnknownArgumentType]
            print(f"{i+1}) {p}")
        
        #get and validate input. repeat until input is valid
        try:
            choice : int = int(input())
        except ValueError:
            print("ERROR: Please enter a number")
        else:
            if choice in range(1, numkeys+1):
                return params[keys[choice-1]]                                                        # pyright: ignore[reportUnknownVariableType]
            else:
                print(f"ERROR: Please enter a number between 1 and {numkeys} (inclusive)")

# Menu specifically for getting the sequence of drops. (WARN: doesn't check if the drops make sense! 
# It is up to the user to ensure that the drops are possible for the piece they want to move)
def drop_menu() -> list[int]: #TODO Make this way better, with its own ui and validation
    i = 1
    output: list[int] = []
    while True:
        print(f"Choose number of drops for tile {i} or -1 to end:")
        
        try:
            choice = int(input())
        except ValueError:
            print("ERROR: Please enter a number")
        else:
            match choice:
                case -1: return output
                case 0: print("ERROR: Please enter a number greater than 0")
                case x if x < 0: print("ERROR: Please enter a non-negative number")
                case _: output.append(choice); i+=1

# Generate a list of possible moves for a given stack in a given direction
def generate_drops(board: "Board", square: tuple[int, int], dir : "Direction", pickup : int) -> list[list[int]]:
    #get the stack to take from and if it has a capstone on top
    stack = board.get_stack(square)
    if stack is None:
        raise IndexError("square out of bounds")
    capstone = stack[-1].piece is PieceType.CAPSTONE

    # get distance to wall from square in direction dir
    wall, distance = board.distance_to_wall(square,dir)
    
    # is the wall hard (capstone or edge of board) or soft (standing stone)
    hard = wall is not PieceType.STANDINGSTONE

    if distance == 0: #handle the case of a single capstone seperately
        if not hard and pickup == 1 and capstone:
            return [[1]] 
    else:
        return gd_cached(pickup,capstone, distance-1,  hard)
    return []

#We can cache the output of this since there are a limited number of possible drops
@functools.cache
def gd_cached(n:int, capstone:bool, dist:int,hard:bool):
    output: list[list[int]] = []
    gd_helper(n,capstone,dist,hard,output,[])
    return output

# Recursively find all possible drops for a given pickup size, n
def gd_helper(n : int, capstone:bool, dist: int, hard:bool, output:list[list[int]], drops: list[int]):
    if n == 0: #if we run out of stones, add the path we used to get here
        output.append(drops.copy())
        return
    elif dist == 0: #if we reach a wall drop all remaining stones (and deal with case where we can squash a standing stone)
        if not hard and capstone and n>1:
            output.append(drops+[n-1, 1])
        output.append(drops+[n])
        return

    #Recurse for every possible amount of stones that could have been dropped
    for i in range(1,n+1):
        drops.append(i)
        gd_helper(n-i, capstone, dist-1, hard, output, drops)
        _=drops.pop() # allows us to use append instead of concat which ends up with less copying


### Piece and Move classes

In [43]:
class Piece:
    __slots__ = ("color", "piece")

    def __init__(self, color : Color, piece_type: PieceType):
        self.color: Color = color
        self.piece: PieceType = piece_type

    #return character with ANSI color applied
    def get_string(self) -> str:
        piece_char = self.piece.name[0]

        match self.color:
            case Color.BLACK: colorcode = W_ON_B
            case Color.WHITE: colorcode = B_ON_W
        
        return colorcode + piece_char + RESET


#Class that represents a move (placement or real move)
#It may be wise to create subclasses for placement and movement
class Move:
    __slots__ = ("square", "direction", "count", "drops", "stone", "flatten")
    def __init__(self,
                 square:tuple[int,int], 
                 direction: Direction | None = None, 
                 count:int = 1, #number of tiles to pick up to move
                 drops:list[int]|None = None, 
                 stone:PieceType = PieceType.FLATSTONE
                 ):
        self.square: tuple[int,int] = square # tuple of values (x, y),
        self.direction: Direction | None = direction
        self.count: int = count
        self.drops: list[int] | None = drops
        self.stone: PieceType | None= stone if direction is None else None
        self.flatten: bool = False

    #return a string that is this move in PTN (Portable Tak Notation)
    def to_ptn(self) -> str:
        square =  chr(97+self.square[0])+str(self.square[1]+1) # Get square rank and column

        # Placement
        if self.direction is None:
            if self.stone is None: raise ValueError("Error while converting piece to PTN: Move cannot have no stone and no direction")
            stone = "" if self.stone is PieceType.FLATSTONE else self.stone.name[0]  
            return stone+square

        # Movement
        else:
            match self.direction:
                case Direction.UP: dir = "+"
                case Direction.RIGHT: dir = ">"
                case Direction.DOWN: dir = "-"
                case Direction.LEFT: dir = "<"

            #count can be excluded if it is 1
            count = "" if self.count == 1 else str(self.count)

            #drops can be excluded if they are the same as count
            if self.drops is not None and self.drops[0] == self.count: 
                drops = ""
            else:
                #drops are always 1 digit numbers because the max board size is 8
                #so it is unambiguous to just concat them all together
                drops = "".join(map(str, self.drops)) if self.drops is not None else ""

            #optional mark for flattening, but helps when trying to reverse moves and debug
            if self.flatten:
                flatten = "*"
            else:
                flatten =  ""

            return count + square + dir + drops + flatten

### Game class

In [44]:
class Game:
    def __init__(self, 
                    p1 : "Player | None" = None,
                    p2 : "Player | None" = None,
                    board_size : int | None = 5,
                    komi : float | None = 0,
                    p1_depth : int = DEPTH,
                    p2_depth : int = DEPTH,
                    board : "Board| None" = None,
                    display_game : bool = False
                 ) : 
        # Get board size if not specified (defaults to 5)
        if board_size is None:
            board_size = menu(
                    { "3x3":3, "4x4":4, "5x5":5, "6x6":6, "7x7":7, "8x8":8},
                    "Please choose a board size:")
        
        # Get komi if not specified (defaults to 0)
        if komi is None:
            komi = menu(
                    {"0":0, "0.5":0.5, "1":1, "1.5":1.5, "2":2, "2.5":2.5, "3":3},
                    "Please choose a komi (1st player handicap):")

        # Get the correct number of stones for the board size
        capstones = CAPSTONES[board_size]
        normal_stones = NORMAL_STONES[board_size]

        # Get P1 and P2 methods (defaults to asking)
        if p1 is None:
            p1= menu({
                "random" : RandomPlayer(), 
                "human": HumanPlayer(), 
                     "minimax": MinimaxPlayer()},
                "Choose a strategy for player 1:")

        p1.__init__(Color.WHITE, normal_stones, capstones, 0, p1_depth, 1)

        if p2 is None:
            p2 = menu({
                "random" : RandomPlayer(), 
                "human": HumanPlayer(), 
                "minimax": MinimaxPlayer()},
                "Choose a strategy for player 2:")

        p2.__init__(Color.BLACK, normal_stones, capstones, komi, p2_depth, 2)

        # Use provided board if it is the correct size, otherwise use empty board.
        self.board: Board = Board(board_size) if board is None or board.size != board_size else board

        self.current_player: Player = p1
        self.current_opponent: Player = p2
        self.display_game: bool = display_game
        self.result_string: str = ""
        self.turn_number : int = 1
        self.moves : list[Move] = []


    # Main loop of the game
    def play(self):
        if self.display_game: self.display()
        # self.opener() #TODO: make minimax work with opener
        winner = self.winner()
        while winner is None:
            if self.display_game: 
                self.display()

            self.turn()
            winner = self.winner()

        if self.display_game:
            print("Game is over!")
            if not winner:
                print("Tie")
            else: print("Winner: Player ",winner)
            self.board.display()

    def display(self):
        #print the turn number, which player's turn it is and what the last move was
        print("="*TERM_WIDTH,"\nTurn ", self.turn_number, "  Current player:", self.current_player.id,"  Previous move: ", self.moves[-1].to_ptn() if self.moves else "n/a")
        #print the board
        self.board.display()
        #print each player, their pieces, their color, and their strategy
        self.current_player.display()
        self.current_opponent.display() 

    #Have each player play a stone as the other player (see self.turn())
    def opener(self):
        self.turn(True)
        if self.display_game: self.display()
        self.turn(True)

    
    #Get a turn from a player and swap players
    def turn(self, opener:bool = False):
        while True: # loop until a valid move is given
            player = self.current_player if not opener else self.current_opponent
            move: Move = self.current_player.get_move(self, opener)
            if self.board.move(move,player): 
                self.moves.append(move)
                break

        #swap players and increase turn num if both have gone
        self.current_player, self.current_opponent = self.current_opponent, self.current_player
        if self.current_player.id == 1:
            self.turn_number += 1


    #Check all win conditions. Return id of winner, 0 for tie, None for not over
    def winner(self) -> int | None: 
        roads = self.board.is_road()
        winner: int | None = None
        result = ""
        # if both players get a road in the same turn, the current_player (one who did it) should win
        if roads[self.current_player.piece_color]:
            if self.display_game: print("Road win!")
            winner = self.current_player.id
            result = "R"

        elif roads[self.current_opponent.piece_color]:
            if self.display_game: print("Road win!")
            winner = self.current_opponent.id
            result = "R"

        elif (self.board.open_spaces() == 0 #out of stones or full board
              or (self.current_player.capstones + self.current_player.normal_stones) == 0
              or (self.current_opponent.capstones + self.current_opponent.normal_stones) == 0):
            result = "F"
            if self.display_game: print("Flat win!")
            counts = self.board.flat_count()
            cur_score = counts[self.current_player.piece_color] + self.current_player.komi
            opp_score = counts[self.current_opponent.piece_color] + self.current_opponent.komi
            if cur_score == opp_score: # only way to tie
                self.result_string = "1/2-1/2"
                return 0
            if cur_score > opp_score:
                winner = self.current_player.id
            else:
                winner = self.current_opponent.id
        if winner == 1:
            self.result_string = result + "-0"
        elif winner == 2:
            self.result_string = "0-" + result
        return winner

    #output the ptn string for the game
    def move_string(self) -> str:
        output = ""
        for i,move in enumerate(self.moves):
            if i%2 == 0:
                output += str((i+2)//2) + ". " + move.to_ptn()
            else:
                output += " " + move.to_ptn() + "\n"

        return output + " " + self.result_string

### Board class

In [45]:
class Board:
    def __init__(self, size: int, board : list[list[list[Piece]]] | None = None):
        self.size: int = size
        if board is None:
            self.grid: list[list[list[Piece]]] = [[[] for _ in range(size)] for _ in range(size)]
        else:
            self.grid = board

    def move(self, move : Move, player: "Player", test : bool = False) -> bool: #TODO: test this function more thuroughly
        if test and not self.test_move(move, player):
            return False



        stack= self.get_stack(move.square)
        if stack is None:
            raise ValueError("None stack while getting stack for move")

        # for placements
        if move.direction is None:
            if move.stone == PieceType.CAPSTONE:
                player.capstones -= 1
            else:
                player.normal_stones -=1

            if move.stone is None:
                raise ValueError("Stone not specified for placement. Move must have either direction or stone.")

            stack.append(Piece(player.piece_color, move.stone))

        # for movements
        else:
            if move.drops is None:
                raise ValueError("Trying to move with None drops")
            pickup = stack[-move.count:]
            assert move.count <= len(stack)
            del stack[-move.count:] #TODO: make sure this does what you think it does

            offset = 1
            while len(pickup) != 0:
                # print("Offset: ", offset)
                # print("Drops: ", move.drops)
                # print("pickup: ", pickup)
                adj_stack = self.get_stack(self.offset_tile(move.square, move.direction, offset))
                if adj_stack is None:
                    raise ValueError("None stack during drops")

                if adj_stack and adj_stack[-1].piece == PieceType.STANDINGSTONE:
                    if pickup[0].piece != PieceType.CAPSTONE:
                        raise ValueError("Attempted to flatten standing stone with non-capstone")
                    adj_stack[-1].piece = PieceType.FLATSTONE
                    move.flatten = True

                # print("extending with: ",pickup[0:move.drops[offset-1]]) 
                adj_stack.extend(pickup[0:move.drops[offset-1]])

                if move.drops[offset-1] < len(pickup):
                    pickup = pickup[move.drops[offset-1]:]
                elif move.drops[offset-1] == len(pickup):
                    pickup = []

                offset += 1 

        return True


    def unmove(self, move : Move, player: "Player"):

        stack = self.get_stack(move.square)
        if stack is None:
            raise ValueError("None stack while geting stack for unmove")

        #For Placements
        if move.direction is None:
            if move.stone == PieceType.CAPSTONE:
                player.capstones += 1 
            else:
                player.normal_stones += 1
            
            if move.stone is None:
                raise ValueError("Stone not specified for un-placement. Move must have either direction or stone.")

            # make sure that the stone is the correct one
            assert stack.pop().piece == move.stone

        #For movements
        else:
            if move.drops is None:
                raise ValueError("Trying to unmove with None drops")
            
            pickup: list[Piece] = []

            
            offset = 1
            for drop in move.drops:
                adj_stack = self.get_stack(self.offset_tile(move.square, move.direction, offset))
                if adj_stack is None:
                    raise ValueError("None stack during undrops")

                dropped = adj_stack[-drop:]

                # dropped.reverse()

                pickup.extend(dropped)

                del adj_stack[-drop:]

                #WARN: this only works if the move has already been used, because that is when
                #move.flatten is set
                if move.flatten and len(dropped) == 1 and dropped[0].piece == PieceType.CAPSTONE:
                    adj_stack[-1].piece = PieceType.STANDINGSTONE
                    move.flatten = False #is this a good thing to do? idk

                offset += 1 

            if len(pickup) != move.count:
                raise ValueError("Number of unpickups based on drops is different from move.count")

            stack.extend(pickup)



            




    
    # return true if a move can be made and false if it can't be
    def test_move(self, move:Move, player: "Player") -> bool: #TODO: don't think this works 
        # (should replace this in the future with just not allowing bad input)
        
        moves = self.enumerate_moves(player)


        if move in moves:
            return True
        else:
            return False

    #return the number of spaces that don't have any piece on them 
    def open_spaces(self) -> int: 
        return sum(1 for x in self.grid for y in x if not y)

    #return a dict with the color of the player as key and the flat count as the value
    def flat_count(self):
        black = sum(1 for x in self.grid for y in x if y and y[-1].color == Color.BLACK and y[-1].piece == PieceType.FLATSTONE)
        white = sum(1 for x in self.grid for y in x if y and y[-1].color == Color.WHITE and y[-1].piece == PieceType.FLATSTONE)
        return {Color.BLACK:black, Color.WHITE:white}

    # return true if there is a roard and false otherwise.
    def is_road(self) :
        #(use dfs from two sides to try to find the other side)
        #(there may be a way to do it incrementally, like store old dfs and use them instead
        #of recalculating every time but idk)

        ub =  self.dfs(Direction.UP, Color.BLACK, (None, 0))
        uw =   self.dfs(Direction.UP, Color.WHITE, (None, 0))
        lb = self.dfs(Direction.LEFT, Color.BLACK, (self.size-1, None))
        lw =   self.dfs(Direction.LEFT, Color.WHITE, (self.size-1, None))

        return {Color.BLACK:ub or lb, Color.WHITE:uw or lw}

    #TODO: test this
    def dfs(self, node: Direction, col: Color, goal:tuple[int|None, int|None] = (None, None)) -> bool:

        stack: list[tuple[int, int]] = []
        visited : set[tuple[int,int]] = set()

        def valid_tile(a:int,b:int) -> bool:
            stk = self.get_stack((a,b))
            assert stk is not None

            return bool(stk) and stk[-1].piece != PieceType.STANDINGSTONE and stk[-1].color == col

        match node:
            case Direction.UP:    stack = [(x, self.size-1) for x in range(0,self.size) if valid_tile(x,self.size-1)] 
            case Direction.RIGHT: stack = [(self.size - 1, y) for y in range(0,self.size) if valid_tile(self.size -1, y)]
            case Direction.DOWN:  stack = [(x, 0) for x in range(0,self.size) if valid_tile(x,0)] 
            case Direction.LEFT:  stack = [(0,y) for y in range(0,self.size) if valid_tile(0,y)] 
        
        while stack:
            n = stack.pop()
            visited.add(n)

            # return true if the traversal reaches the goal row/column
            if n[1] == goal[1] or n[0] == goal[0]:
                return True

            # add neighbors to stack if they havn't been visited
            for i in range(4):
                neighbor = self.offset_tile(n, DIRECTIONS[i])
                if max(neighbor) >= self.size or min(neighbor) < 0: continue
                tile = self.get_stack(neighbor)
                if tile: 
                    p = tile[-1]
                    if p.color == col and p.piece != PieceType.STANDINGSTONE and neighbor not in visited:
                        stack.append(neighbor)

        # if the stack is empty after trying to add stuff to it, no path exists 
        return False

    #pretty print the board
    def display(self) -> None:
        global_max = max(max(len(z) for x in self.grid for z in x), 10)  #max height of any stack
        p = min(global_max,((TERM_WIDTH)-1)//self.size-1) # number of chars of padding on each side of stack characters
        top = " ╭"+ ("─"*(2*p+1)+"┬")*(self.size-1) +"─"*(2*p+1)+"╮"
        mid = " ├"+ ("─"*(2*p+1)+"┼")*(self.size-1) +"─"*(2*p+1)+"┤"
        bot = " ╰"+ ("─"*(2*p+1)+"┴")*(self.size-1) +"─"*(2*p+1)+"╯"
        letters = " "
        for i in range(self.size):
            letters = letters + " "*(p+1)+chr(97+i)+" "*(p) #letter label for bottom


        print(top)
        for i,row in enumerate(self.grid):
            max_height = global_max
            row_str = ""
            if max_height == 0: # handle all cels empty case
                row_str = (" "*(2*p+1)+"")*(self.size+1)
            else:
                while max_height > 0:
                    if max_height == global_max//2:
                        line_str = str(self.size - i)+"│" # draw index numbers
                    else:
                        line_str = " │"

                    for stack in row: 
                        if len(stack) >= max_height: # print things with tiles at this height
                            line_str += " "*p+ stack[max_height-1].get_string() + " "*p+"│"
                        else: # print spaces for things not at this height
                            line_str += " "*(2*p+1)+"│"
                    max_height -= 1
                    row_str = row_str + line_str + "\n"
            if i != 0: print(mid)
            print(row_str[:-1])
        print(bot)
        print(letters)


    #return a list of all valid moves for the player in current position
    def enumerate_moves(self, player : "Player", opener:bool = False) -> list[Move]:
        moves: list[Move] = []
        # for every square...
        for row in range(self.size):
            for col in range(self.size):
                square = (col,row)
                stack = self.get_stack(square)
                if not stack: # ... if its empty, the player can place any stone they have on it ...
                    if player.normal_stones > 0:
                        moves.append(Move(square)) #(only flatstone placement allowed in the opener)
                        if not opener: moves.append(Move(square,stone=PieceType.STANDINGSTONE))
                    if player.capstones > 0 and not opener:
                        moves.append(Move(square, stone=PieceType.CAPSTONE))
                elif stack[-1].color == player.piece_color and not opener: # ... and if it is their color they can move it.
                    for pickup in range(1,min(self.size, len(stack))+1):
                        for i in range(4):
                            dir = DIRECTIONS[i]
                            possible_drops = generate_drops(self, square, dir, pickup)
                            for drops in possible_drops:
                                moves.append(Move(square,dir,pickup, drops)) 
        if RANDOM_MOVE_ORDERING: random.shuffle(moves)
        return moves

    # gives the tile "times" number of tiles in "dir" direction
    #TODO: doesn't need to be a method of board..
    #PERF: one of the most called functions
    def offset_tile(self, tile:tuple[int,int], dir: Direction,  times:int = 1) -> tuple[int,int]:
        match dir:
            case Direction.UP:    return (tile[0], tile[1]+times)
            case Direction.RIGHT: return (tile[0]+times, tile[1])
            case Direction.DOWN:  return (tile[0], tile[1]-times)
            case Direction.LEFT:  return (tile[0]-times, tile[1])

    # Returns the stack on the given tile. Returns None if the tile is outside the board area.
    #PERF: this is the most called function in the program...
    def get_stack(self, tile : tuple[int, int]) -> list[Piece] | None:
        #NOTE: Tile tuple has ints in range [0,size-1] inclusive
        if max(tile) >= self.size or min(tile) < 0: return None
        else: return self.grid[self.size - 1 - tile[1]][tile[0]]
    
    #returns a tuple in the form (type of wall (None for edge of board),  distance to wall)
    def distance_to_wall(self,tile:tuple[int,int], dir : Direction) -> tuple[PieceType | None, int]:
        for i in range(0,self.size):
            stack = self.get_stack(self.offset_tile(tile, dir,i+1))
            if stack is None: return (None, i) # wall
            if stack:
                match stack[-1].piece:
                    case PieceType.FLATSTONE: pass
                    case x: return (x, i)
        return (None, 0)

### Player classes

In [46]:
#base player class
class Player:
    def __init__(self,
                 piece_color : Color = Color.BLACK,
                 normal_stones : int = 0,
                 capstones : int = 0, komi : float = 0,
                 depth: int = DEPTH,
                 id : int = 0,
                 ab : bool = ALPHA_BETA
                 ):
        self.piece_color: Color = piece_color
        self.komi: float = komi
        self.normal_stones: int = normal_stones
        self.capstones: int = capstones
        self.depth: int = depth
        self.id: int = id
        self.ab: bool = ab
        
    def get_move(self, game : Game, opener:bool = False) -> "Move":   # pyright: ignore[reportUnusedParameter]
        raise NotImplementedError

    def display(self) -> None:
        if self.piece_color == Color.BLACK:
            color = W_ON_B
        else:
            color = B_ON_W
        strat = self.player_type()
        print(f"{color}Player:{self.id}  Strategy:{strat}, Normal Stones: {self.normal_stones}  Capstones: {self.capstones}{RESET}")

    def player_type(self) ->str:
        raise NotImplementedError



class HumanPlayer(Player):
    @override
    #get move from player by asking for each part
    def get_move(self, game: Game, opener:bool = False) -> Move:
        #TODO:: add checking for each part. Add option during start of game to use PTN

        if opener: print("OPENER: choose a square to place one of your opponent's flatstones on!")

        #Get square
        rows = {chr(ord('A')+x):x for x in range(0,game.board.size)}
        cols = {str(x+1):x for x in range(0,game.board.size)}
        square: tuple[int, int] = ( 
            menu(rows, "Choose a column:"),
            menu( cols, "Choose a row:")
        )

        if opener: return Move(square, stone = PieceType.FLATSTONE)

        stack = game.board.get_stack(square)
        if not stack:  #choose piece for placement
            piece = menu({"flatstone":PieceType.FLATSTONE, 
                          "standing stone":PieceType.STANDINGSTONE,
                          "capstone stone":PieceType.CAPSTONE}, "What kind of piece would you like to place?:")
            return Move(square,stone = piece)

        else: #movement

            direction = menu({"up":Direction.UP, "right":Direction.RIGHT,
                              "down":Direction.DOWN, "left":Direction.LEFT},
                             "Chose a direction to move:")
            count = 1
            if len(stack) > 1:
                count = menu({str(x):x for x in range(1,min(game.board.size, len(stack))+1)}, 
                             "How many pieces do you want to grab from this stack?")

            drops = drop_menu()

            return Move(square, direction, count, drops)

    @override
    def player_type(self) -> str:
        return "Human"



class RandomPlayer(Player):
    @override
    #select random move from list of all possible moves
    def get_move(self, game: Game, opener:bool = False) -> Move:
        moves = game.board.enumerate_moves(self, opener)
        return random.choice(moves)

    @override
    def player_type(self) -> str:
        return "Random"

class MinimaxPlayer(Player):    

    @override
    def get_move(self, game : Game, opener:bool = False) -> Move:#TODO:
        assert self.depth is not None
        v, move = self.minimax(game.board,self.depth,game.current_opponent, True, -inf if self.ab else None, inf if self.ab else None)
        # print("move: ", move.to_ptn(), " value: ", v)
        return move

    #TODO:  There are still some bugs (3 look ahead looses to 2 lookahead on a 3x3 board?)
    # also would be a good idea to introduce some randomness if there are multiple equally good moves
    # also should probably disincentivise making super tall stacks but idk...
    # absolutely need to optimize much better (and do alpha-beta...)
    # find a way to make the agent not give up (currently the slight penalty for longer games means that 
    # if all paths would lead to the opponent winning based on the agent's play, it will end the game asap,
    # but it should instead try to prolong the game to see if the the opponent makes a mistake)
    def minimax(self, board:Board, max_depth:int, op:Player, own_turn:bool, alpha: float|None = None, beta: float|None = None) -> tuple[float, Move]:

        #check if game is over. if so, return evaluation of current board
        roads = board.is_road()
        if (roads[self.piece_color] 
            or roads[op.piece_color]
            or self.capstones + self.normal_stones == 0 
            or op.capstones + op.normal_stones == 0
            or board.open_spaces() == 0
            or max_depth == 0):
            return (self.evaluate_board(board, 0, own_turn), Move((-1,-1))) #TODO: Komi

        best_moves: list[tuple[float,Move]] = [] #TODO: use a priority Queue and pop into a list until one of the moves is lower scored, then choose randomly from that


        player = self if own_turn else op
        possible_moves = board.enumerate_moves(player)

        best_value = float('-inf') if own_turn else float('inf')

        for move in possible_moves: 
            # we move, recurse and unmove. This is much faster than copying each time
            _=board.move(move, player)
            v,_ = self.minimax(board, max_depth-1, op, not own_turn, alpha, beta)
            board.unmove(move, player)

            #if self.depth == max_depth:
            #    print(move.to_ptn(), ":", v)

            v*=MINIMAX_DECAY

            #WARN: it is important that these are in this order for alpha-beta. This is because
            #moves will return their best/worst value bounded by alpha and beta, so they may be tied
            #with the best move but actually have a worse value. The earliest instance of a value is always valid though.
            min_max = max if own_turn else min 
            if v == best_value:
                best_moves.append((v, move))
            elif min_max(v,  best_value) == v:
                best_moves =[(v,move)]
                best_value = v


            if alpha is not None and beta is not None:
                if own_turn:
                    if v >= beta:
                        break
                    alpha = max(alpha, best_moves[0][0])
                else:
                    if v <= alpha:
                        break
                    beta = min(beta, best_moves[0][0])

            

        # can also return random choice NOT FOR ALPHA BETA YOU CANT I DONT THINK
        return best_moves[0] 


            

        
    #evaluate the board based on what I think is important
    def evaluate_board(self, board:Board, komi:float, own_turn:bool, ) -> float:
        op_color = Color((self.piece_color.value + 1)%2)
        flat_count = board.flat_count() #dict color  -> int
        flat_score = flat_count[self.piece_color]-flat_count[op_color]
        
        #TODO: test different values for roads (maybe make based on board size?)
        road = board.is_road()
        road_score = 0
        if road[self.piece_color]:
            road_score += 100
        if road[op_color]:
            road_score -= 100
            if road_score == 0: # if we make the move we win otherwise we lose
                road_score += -100 if not own_turn else 100 

        # Ideas: Hard caps, discourage stacks above carry limit, add small bonus for walls
        # add small punishment every turn to force offensive play, add bonus for the number of 
        # posible moves (and punish giving op more move options)
        
        return flat_score if road_score == 0 else road_score

    #TODO: iterative deepening


            
    @override
    def player_type(self) -> str:
        return "Minimax (Depth:" + str(self.depth) + ")"


### Examples
Choose options for the games below, run the cell that contains the game that you want to try, and then run the final cell to run the game. 

If you use the human interface, be warned that is not the most user friendly and will allow several types of illegal moves (moving the other players pieces, placing pieces that you don't have) and several things that will cause exceptions (trying to drop more or less pieces than picked up, dropping pieces outside the board area etc.) and is thus predicated on some knowledege of the game (and even then can be somewhat confusing).

In [47]:
#OPTIONS:
#should the games print the state on each move?
PRINT_BOARD = True
#should the games print the board after finishing?
PRINT_FINAL_BOARD = True
#should the game print the game notation once the game is over?
PRINT_PTN = True
#Size of the board
BOARD_SIZE = 5
#2nd player score bonus WARNING: non-zero komi is not properly handled by minimax
KOMI = 0

Example 1: Random vs Random 

In [48]:
game = Game(RandomPlayer(), RandomPlayer(), BOARD_SIZE, KOMI, DEPTH, DEPTH, None, PRINT_BOARD)

Example 2: Human vs Random

In [49]:
game = Game(HumanPlayer(), RandomPlayer(), BOARD_SIZE, KOMI, DEPTH, DEPTH, None, PRINT_BOARD)

Example 3: Minimax vs Random

In [50]:
game = Game(MinimaxPlayer(), RandomPlayer(), BOARD_SIZE, KOMI, DEPTH, DEPTH, None, PRINT_BOARD)

Example 4: Minimax vs Human

In [51]:
game = Game(MinimaxPlayer(), HumanPlayer(), BOARD_SIZE, KOMI, DEPTH, DEPTH, None, PRINT_BOARD)

Example 5: Minimax vs Minimax

In [52]:
game = Game(MinimaxPlayer(), MinimaxPlayer(), BOARD_SIZE, KOMI, DEPTH, DEPTH, None, PRINT_BOARD)

Example 6: Minimax vs Minimax+1

In [53]:
game = Game(MinimaxPlayer(), MinimaxPlayer(), BOARD_SIZE, KOMI, DEPTH, DEPTH+1, None, PRINT_BOARD)

Example 7: Minimax+1 vs Minimax

In [54]:
game = Game(MinimaxPlayer(), MinimaxPlayer(), BOARD_SIZE, KOMI, DEPTH+1, DEPTH, None, PRINT_BOARD)

Example 8: Game creation functionality

In [15]:
game = Game(None, None, None, None, DEPTH, DEPTH, None, PRINT_BOARD)

Please choose a board size:
1) 3x3
2) 4x4
3) 5x5
4) 6x6
5) 7x7
6) 8x8


 1


Please choose a komi (1st player handicap):
1) 0
2) 0.5
3) 1
4) 1.5
5) 2
6) 2.5
7) 3


 1


Choose a strategy for player 1:
1) random
2) human
3) minimax


 1


Choose a strategy for player 2:
1) random
2) human
3) minimax


 1


**PLAY GAME**:

In [55]:
game.play()
if PRINT_FINAL_BOARD and not PRINT_BOARD:
    game.display()
if PRINT_PTN:
    print(game.move_string())

Turn  1   Current player: 1   Previous move:  n/a
 ╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────╮
 │                 │                 │                 │                 │                 │
 │                 │                 │                 │                 │                 │
 │                 │                 │                 │                 │                 │
 │                 │                 │                 │                 │                 │
 │                 │                 │                 │                 │                 │
5│                 │                 │                 │                 │                 │
 │                 │                 │                 │                 │                 │
 │                 │                 │                 │                 │                 │
 │                 │                 │                 │                 │                 │
 │                 │